# Advanced Type Hinting & Annotation Management

A comprehensive guide to Python's type system — from precise domain modeling to annotation-driven frameworks.  
All examples use the **Globomantics Telemetry Hub** as a running case study.

## Contents

1. [How Python's Type System Works](#1-how-pythons-type-system-works)
2. [Mastering Generics and Callable Patterns](#2-mastering-generics-and-callable-patterns)
3. [Building Protocol-Oriented Architectures](#3-building-protocol-oriented-architectures)
4. [Orchestrating Annotations at Runtime](#4-orchestrating-annotations-at-runtime)
5. [Managing Annotations Across a Large Codebase](#5-managing-annotations-across-a-large-codebase)
6. [Typing Tooling, CI, and Developer Experience](#6-typing-tooling-ci-and-developer-experience)
7. [Designing an Annotation-Driven Framework](#7-designing-an-annotation-driven-framework)

---
## 1. How Python's Type System Works

### The three layers of the type system

```
┌──────────────────┬──────────────────────┬──────────────────────┐
│   Annotations    │    Type checkers     │    The runtime       │
├──────────────────┼──────────────────────┼──────────────────────┤
│ Just syntax to   │ External tools:      │ Stores annotations   │
│ Python           │ mypy and pyright     │ for tools to read    │
│                  │                      │                      │
│ Parsed and       │ All compatibility    │ Never enforces them  │
│ attached to      │ rules live here      │                      │
│ objects, nothing │                      │ Nothing checked      │
│ more             │ Semantics built      │ between bad data     │
│                  │ outside the language │ and the crash        │
└──────────────────┴──────────────────────┴──────────────────────┘
```

**Key insight:** Python's type system is *gradual*. Typed and untyped code coexist. The checker enforces rules only where annotations exist, and trusts the rest. `Any` is the escape hatch that connects both worlds.

### Gradual typing

- Millions of lines of untyped Python already existed when typing was introduced
- Typed and untyped code coexist in one program
- Checked where annotations exist, **trusted** where they don't
- `Any` is the escape hatch — it is compatible with every type in **both directions**

### `Any` vs `object`

```
┌────────────────────────────────┬──────────────────────────────────┐
│            Any                 │            object                │
├────────────────────────────────┼──────────────────────────────────┤
│ Compatible with every type,    │ The honest top type              │
│ in both directions             │                                  │
│                                │ Everything is assignable to it,  │
│ Every operation passes,        │ not the reverse                  │
│ inference poisoned downstream  │                                  │
│                                │ Only operations every object     │
│ FileSource returned Any, so    │ supports are allowed             │
│ nothing downstream was checked │                                  │
│                                │ Using it as something specific   │
│                                │ requires proof                   │
└────────────────────────────────┴──────────────────────────────────┘
```

**Rule:** Use `object` when you truly mean "any value". Reserve `Any` for quarantining untyped boundaries.

### Nominal vs structural subtyping

```
┌──────────────────────────────┬──────────────────────────────────┐
│           Nominal            │           Structural             │
├──────────────────────────────┼──────────────────────────────────┤
│ Goes by declared names       │ Goes by shape                    │
│                              │                                  │
│ A class is a subtype of      │ Right methods and signatures fit,│
│ what it inherits from        │ no inheritance needed            │
│                              │                                  │
│                              │ The basis for Protocol           │
└──────────────────────────────┴──────────────────────────────────┘
```

### Assignability

> Every checker error is this question answered **no**: *Can a value of this type flow into a slot of that type?*

### Type narrowing

Type narrowing makes a type more precise inside a branch:
- `isinstance` checks, comparisons, and truthiness tests narrow automatically
- `TypeIs` lets you write user-defined narrowing functions
- Narrowing is what makes union types practical

### The modern annotation baseline

```
┌─────────────────────────────────┬──────────────────────────────────┐
│         Modern form             │         Legacy spelling          │
├─────────────────────────────────┼──────────────────────────────────┤
│ X | None                        │ Optional[X]                      │
│ list[str]                       │ typing.List[str]                 │
│ The type statement              │ TypeAlias assignments            │
│ TypeIs functions                │ Most TypeGuard uses              │
└─────────────────────────────────┴──────────────────────────────────┘
```

Legacy spellings still work and persist in real codebases (for Python < 3.10 compatibility). Read both, write the modern form.

### Domain-modeling tools

```
┌──────────────────────┬──────────────────────┬──────────────────────┐
│      TypedDict       │       NewType        │       TypeIs         │
├──────────────────────┼──────────────────────┼──────────────────────┤
│ A type for           │ A distinct identity  │ User-written         │
│ dict-shaped data     │ on an existing type  │ narrowing functions  │
│                      │                      │                      │
│ Declared keys, a     │ A device ID stays a  │ Tell the checker     │
│ value type for each  │ plain str at runtime │ what a passing       │
│                      │                      │ validation proves    │
│ Fits the raw wire    │ The checker keeps it │                      │
│ payloads             │ apart from other     │ The narrowing        │
│                      │ strings              │ function for the     │
│                      │                      │ parse boundary       │
└──────────────────────┴──────────────────────┴──────────────────────┘
```

### Demo: Before and after — typing the telemetry domain

**Before (untyped):** `Any` everywhere — the checker can't catch a string passed as temperature.

In [1]:
# BEFORE: untyped domain — Any poisons downstream inference
from typing import Any

def is_valid_payload(payload: Any) -> bool:  # payload is Any — no narrowing
    return isinstance(payload, dict)

def summarize(readings: Any) -> Any:         # Any in, Any out — checker goes silent
    return {"total": len(readings)}

**After (typed):** Precise domain model — every boundary is checkable.

In [2]:
# AFTER: fully typed domain model
from dataclasses import dataclass
from typing import Literal, NewType, TypedDict, TypeIs

# PEP 695 type alias (Python 3.12+)
type Status = Literal["ok", "warning", "error"]

# NewType: distinct identity — DeviceId is a str at runtime, but the checker
# keeps it apart from arbitrary strings
DeviceId = NewType("DeviceId", str)

REQUIRED_FIELDS = ("device_id", "temperature", "status", "timestamp")
KNOWN_STATUSES: tuple[Status, ...] = ("ok", "warning", "error")


# TypedDict: models the raw wire payload (dict-shaped, unvalidated)
class RawPayload(TypedDict):
    device_id: str
    temperature: float
    status: str
    timestamp: str


# Immutable domain object after validation
@dataclass(frozen=True)
class Reading:
    device_id: DeviceId
    temperature: float
    status: Status
    timestamp: str


# TypedDict for the summary result
class Summary(TypedDict):
    total: int
    by_status: dict[Status, int]
    average_temperature: float


# TypeIs: user-written narrowing — tells the checker that if this returns True,
# the payload is a RawPayload
def is_valid_payload(payload: object) -> TypeIs[RawPayload]:
    if not isinstance(payload, dict):
        return False
    if any(field not in payload for field in REQUIRED_FIELDS):
        return False
    return (
        isinstance(payload["device_id"], str)
        and type(payload["temperature"]) in (int, float)
        and isinstance(payload["status"], str)
        and isinstance(payload["timestamp"], str)
    )


def is_known_status(value: str) -> TypeIs[Status]:
    return value in KNOWN_STATUSES


def parse_reading(payload: RawPayload) -> Reading:
    status = payload["status"]
    if not is_known_status(status):
        raise ValueError(f"unknown status: {status!r}")
    return Reading(
        device_id=DeviceId(payload["device_id"]),
        temperature=payload["temperature"],
        status=status,
        timestamp=payload["timestamp"],
    )


def summarize(readings: list[Reading]) -> Summary:
    total = len(readings)
    by_status: dict[Status, int] = {}
    temperatures: list[float] = []
    for reading in readings:
        by_status[reading.status] = by_status.get(reading.status, 0) + 1
        temperatures.append(reading.temperature)
    average = sum(temperatures) / total if total else 0.0
    return {
        "total": total,
        "by_status": by_status,
        "average_temperature": round(average, 2),
    }


# Demonstrate narrowing
raw: object = {"device_id": "dev-01", "temperature": 22.5, "status": "ok", "timestamp": "2024-01-01"}

if is_valid_payload(raw):
    # Inside this branch, raw is narrowed to RawPayload
    reading = parse_reading(raw)
    print(reading)

results = summarize([Reading(DeviceId("dev-01"), 22.5, "ok", "2024-01-01")])
print(results)

Reading(device_id='dev-01', temperature=22.5, status='ok', timestamp='2024-01-01')
{'total': 1, 'by_status': {'ok': 1}, 'average_temperature': 22.5}


---
## 2. Mastering Generics and Callable Patterns

### Why generics?

The pipeline has stages to chain, reorder, and extend. Without generics:
- Nothing stops a wrong order or a skipped stage
- A mischained stage crashes at runtime, in production
- The `Any` shortcut makes the problem silent until production

### Type parameters with PEP 695 (Python 3.12+)

```python
# Old style (pre-3.12): manual TypeVar, imported and threaded by hand
from typing import TypeVar
T = TypeVar("T")
def first(items: list[T]) -> T: ...

# PEP 695: declared inline, in square brackets — no imports, no scoping puzzles
def first[T](items: list[T]) -> T: ...
class Stage[T]: ...
```

### Bounds vs constraints

```
┌────────────────────────────────┬──────────────────────────────────┐
│        Bound (T: Comparable)   │    Constraints (T: (str,bytes))  │
├────────────────────────────────┼──────────────────────────────────┤
│ Accepts the bound and every    │ Accepts exactly the listed types │
│ subtype                        │                                  │
│                                │ Choose when the set is closed    │
│ Choose when the types share    │ and known                        │
│ behavior                       │                                  │
└────────────────────────────────┴──────────────────────────────────┘
```

### Variance

- **Covariant** (containers you read from): a `Source[Reading]` is a `Source[object]` — narrower type is a subtype
- **Contravariant** (containers you write to): a `Sink[object]` serves as a `Sink[Reading]` — wider type is a subtype
- PEP 695 classes infer variance automatically; manual `TypeVar` needs `covariant=True` flags

### TypeVar defaults (PEP 696)

```python
class Pipeline[TIn, TOut = TIn]: ...

Pipeline[list[Reading]]   # = Pipeline[list[Reading], list[Reading]]
```

Unless stated, the output type is the input type. The general two-parameter form stays expressible.

### The honest-callables toolkit

```
┌──────────────────────────────────────────────────────────────────────┐
│  Callable[..., T]  →  lazy ellipsis: any call accepted, checker off  │
│  Callable[[A,B],R] →  precise: exactly (A, B) -> R                  │
│  Callable[P, R]    →  ParamSpec: captures whole parameter list       │
│  Callable[Concatenate[Arg,P], R] → shifts leading arg               │
│  @overload         →  each call shape gets its own return type       │
└──────────────────────────────────────────────────────────────────────┘
```

### The decorator signature problem

`def decorator(fn: Callable[..., T]) -> Callable[..., T]` makes every wrapped function a black box. Autocomplete, argument checks, and renames die at the `@` line. **`ParamSpec`** solves this.

### `ParamSpec`

```python
def instrument[**P, R](func: Callable[P, R]) -> Callable[P, R]: ...
```

- Captures the whole parameter list as one variable
- `Callable[P, R]` in, `Callable[P, R]` out
- Wrapped functions keep their exact signatures

### `Concatenate`

```python
Callable[Concatenate[Arg, P], R]
```

For wrappers that add or consume a leading argument. One known first parameter, then everything `P` captured.

In [3]:
# Generic Pipeline — the chaining machinery typed for any payload
import functools
from collections.abc import Callable, Iterable

# Generic type alias: a stage is a callable from TIn to TOut
type Stage[TIn, TOut] = Callable[[TIn], TOut]


class Pipeline[TIn, TOut = TIn]:
    """Type-safe composable pipeline."""

    def __init__(self, stage: Stage[TIn, TOut]) -> None:
        self._stage = stage

    def then[TNext](self, stage: Stage[TOut, TNext]) -> "Pipeline[TIn, TNext]":
        """Chain another stage; the checker tracks the new output type."""
        first = self._stage
        def combined(value: TIn) -> TNext:
            return stage(first(value))
        return Pipeline(combined)

    def run(self, value: TIn) -> TOut:
        return self._stage(value)


# Signature-preserving retry decorator using ParamSpec
def retry[**P, R](times: int = 3) -> Callable[[Callable[P, R]], Callable[P, R]]:
    """Retry decorator that preserves the wrapped function's exact signature."""
    def decorator(func: Callable[P, R]) -> Callable[P, R]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            last_exc: Exception | None = None
            for _ in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception as exc:
                    last_exc = exc
            raise last_exc  # type: ignore[misc]
        return wrapper
    return decorator


# Build a typed pipeline
def double(x: int) -> int:
    return x * 2

def to_str(x: int) -> str:
    return str(x)

pipeline = Pipeline(double).then(to_str)  # inferred: Pipeline[int, str]
print(pipeline.run(5))   # "10"


# Overloaded API — return type depends on arguments
from typing import Literal, overload

class DataStore:
    def __init__(self) -> None:
        self._data: list[int] = [10, 20, 30]

    @overload
    def get(self, key: str) -> list[int]: ...
    @overload
    def get(self, key: str, *, latest: Literal[True]) -> int | None: ...
    def get(self, key: str, *, latest: bool = False) -> list[int] | int | None:
        if latest:
            return self._data[-1] if self._data else None
        return self._data

store = DataStore()
all_values = store.get("x")             # inferred: list[int]
latest_value = store.get("x", latest=True)  # inferred: int | None
print(all_values, latest_value)

10
[10, 20, 30] 30


---
## 3. Building Protocol-Oriented Architectures

### Nominal (ABC) vs structural (Protocol) contracts

```
┌──────────────────────────────────┬──────────────────────────────────┐
│           ABC (Nominal)          │        Protocol (Structural)     │
├──────────────────────────────────┼──────────────────────────────────┤
│ Contract enforced by inheritance │ Contract enforced by shape       │
│                                  │                                  │
│ class Sink(ABC):                 │ class Sink(Protocol):            │
│     @abstractmethod              │     def write(self, ...) -> None │
│     def write(self, ...) -> None │                                  │
│                                  │ Any class with .write() fits,    │
│ Implementors must inherit ABC    │ no inheritance needed            │
│                                  │                                  │
│ Couples all implementations      │ Decouples producers from         │
│ to one import tree               │ consumers entirely               │
└──────────────────────────────────┴──────────────────────────────────┘
```

### `@runtime_checkable` — what it provides vs what it cannot guarantee

```
┌──────────────────────────────────────────────────────────────┐
│                    @runtime_checkable                        │
├──────────────────────────┬───────────────────────────────────┤
│        Provides          │       Cannot guarantee            │
├──────────────────────────┼───────────────────────────────────┤
│ isinstance(x, Protocol)  │ Method signatures match           │
│ checks method presence   │                                   │
│                          │ Generic parameter correctness     │
│                          │ (Sink[Summary] ≠ Sink[Reading]    │
│                          │ at runtime — generics are erased) │
└──────────────────────────┴───────────────────────────────────┘
```

**The runtime trap:** `isinstance(bad, Sink)` returns `True` even if `bad.write` has the wrong signature. The static checker catches the mismatch at compile time; `isinstance` only checks method *presence*.

In [4]:
from typing import Protocol, runtime_checkable


# Source protocol: anything with a .readings() method fits
class ReadingSource(Protocol):
    def readings(self) -> list[object]: ...


# Generic protocol: Sink[T] — anything that can write items of type T
@runtime_checkable
class Sink[T](Protocol):
    def write(self, item: T, /) -> None: ...


# Type alias for a specific sink
type ReadingSink = Sink[dict]  # simplified — in the real app, Sink[Summary]


# A concrete sink — no inheritance required
class ConsoleSink:
    def write(self, item: dict, /) -> None:
        print(f"[sink] {item}")

# A bad sink — wrong signature
class BadSink:
    def write(self, item: dict, extra: str) -> None:  # wrong: extra param
        pass

good = ConsoleSink()
bad = BadSink()

# isinstance only checks method *presence*, not signature
print(f"isinstance(good, Sink): {isinstance(good, Sink)}")  # True
print(f"isinstance(bad,  Sink): {isinstance(bad,  Sink)}")   # True  ← runtime trap!

# The static checker would flag: bad has wrong write() signature
# sink: Sink[dict] = bad   ← mypy/pyright error caught here, not at isinstance

good.write({"total": 5})

isinstance(good, Sink): True
isinstance(bad,  Sink): True
[sink] {'total': 5}


---
## 4. Orchestrating Annotations at Runtime

### Three eras of annotation evaluation

```
  Originally (3.0–3.9)      PEP 563 (3.10+)           PEP 649/749 (3.14+)
  ──────────────────────    ──────────────────────    ──────────────────────
  Annotations evaluated     from __future__ import    Deferred evaluation
  eagerly at class/fn       annotations               by default
  definition time           makes all annotations     Evaluated lazily only
                            strings at parse time     when accessed
  NameError if a type       __annotations__ stores    annotationlib provides
  isn't defined yet         raw strings, not objects  Format enum to control
                                                       evaluation
```

### Three `annotationlib` formats (Python 3.14)

```
┌──────────────────────┬──────────────────────┬──────────────────────┐
│     Format.VALUE     │   Format.FORWARDREF  │   Format.STRING      │
├──────────────────────┼──────────────────────┼──────────────────────┤
│ Evaluates the        │ Returns ForwardRef   │ Raw source strings,  │
│ annotation to get    │ objects for names    │ never evaluated      │
│ the actual type      │ that aren't defined  │                      │
│                      │ yet                  │ Safe for tooling     │
│ Raises NameError if  │                      │ and documentation    │
│ a type isn't defined │ Safe mid-load        │ generators           │
│                      │ Avoids NameError     │                      │
└──────────────────────┴──────────────────────┴──────────────────────┘
```

In [5]:
# annotationlib is Python 3.14+; show the pattern with get_type_hints for 3.10+
# The API and concepts are the same.
import typing
from dataclasses import dataclass
from typing import Literal, NewType, get_type_hints, get_origin, get_args, Annotated

type Status = Literal["ok", "warning", "error"]
DeviceId = NewType("DeviceId", str)

@dataclass(frozen=True)
class Reading:
    device_id: DeviceId
    temperature: float
    status: Status
    timestamp: str

# get_type_hints() resolves string annotations to actual types
hints = get_type_hints(Reading)
print("Resolved hints:", hints)

# Direct __annotations__ gives raw strings under PEP 563
print("Raw __annotations__:", Reading.__annotations__)

Resolved hints: {'device_id': __main__.DeviceId, 'temperature': <class 'float'>, 'status': Status, 'timestamp': <class 'str'>}
Raw __annotations__: {'device_id': __main__.DeviceId, 'temperature': <class 'float'>, 'status': Status, 'timestamp': <class 'str'>}


In [6]:
# Annotation-driven validation engine
# Walks a dataclass's annotations and validates a payload against them

from typing import Union
import types


def _check(path: str, value: object, annotation: object) -> list[str]:
    """Recursively validate value against annotation, return error messages."""
    origin = get_origin(annotation)
    args = get_args(annotation)

    # Handle NewType: the __supertype__ attribute points to the underlying type
    if callable(annotation) and hasattr(annotation, "__supertype__"):
        return _check(path, value, annotation.__supertype__)

    # Handle TypeAliasType (Python 3.12 'type' statement)
    if isinstance(annotation, type(type(None))) or isinstance(annotation, type):
        if not isinstance(value, annotation):
            return [f"{path}: expected {annotation.__name__}, got {type(value).__name__}"]
        return []

    # Handle Literal
    if origin is Literal:
        if value not in args:
            return [f"{path}: {value!r} not in {args}"]
        return []

    # Handle Union (X | Y)
    if origin is Union or origin is types.UnionType:
        for arg in args:
            if not _check(path, value, arg):
                return []
        return [f"{path}: {value!r} doesn't match any of {args}"]

    return []


def validate_payload(model: type, payload: object) -> list[str]:
    """Validate a raw payload dict against a dataclass model's annotations."""
    if not isinstance(payload, dict):
        return [f"payload: expected dict, got {type(payload).__name__}"]

    errors: list[str] = []
    hints = get_type_hints(model)

    for field_name, annotation in hints.items():
        if field_name not in payload:
            errors.append(f"{field_name}: missing")
            continue
        errors.extend(_check(field_name, payload[field_name], annotation))

    return errors


# Demonstrate
good_payload = {"device_id": "dev-01", "temperature": 22.5, "status": "ok", "timestamp": "2024-01-01"}
bad_payload  = {"device_id": "dev-01", "temperature": "hot", "status": "boom", "timestamp": "2024-01-01"}

print("Good:", validate_payload(Reading, good_payload))
print("Bad: ", validate_payload(Reading, bad_payload))

Good: []
Bad:  ['temperature: expected float, got str']


---
## 5. Managing Annotations Across a Large Codebase

### Stub files (`.pyi`) for untyped dependencies

When a library has no type annotations, create a stub file:

```python
# legacy_ingest/__init__.pyi  — stub for an untyped third-party library
def fetch_batch(source: str) -> list[dict[str, object]]: ...
def push_record(record: dict[str, object], destination: str) -> bool: ...
```

Stub files contain only signatures (no bodies). They are `.pyi` files alongside (or instead of) the `.py` source.

### Stubs vs inline types

```
┌──────────────────────────────────┬──────────────────────────────────┐
│          Stub files              │        Inline types              │
├──────────────────────────────────┼──────────────────────────────────┤
│ For code you don't own           │ For code you own                 │
│                                  │                                  │
│ No runtime overhead              │ Co-located with the code         │
│                                  │                                  │
│ Can describe third-party libs    │ Checked by mypy directly         │
│                                  │                                  │
│ typeshed provides stubs for      │ Better IDE support               │
│ the standard library             │                                  │
└──────────────────────────────────┴──────────────────────────────────┘
```

### `py.typed` marker

Add an empty `py.typed` file to your package to signal to type checkers that the package is fully typed (PEP 561):
```
mypackage/
    __init__.py
    py.typed      ← empty marker file
    models.py
```

### mypy strict mode with per-module relaxation

```toml
# pyproject.toml
[tool.mypy]
strict = true
enable_error_code = ["deprecated"]

# Relax rules for legacy/third-party modules only
[[tool.mypy.overrides]]
module = ["legacy_ingest", "legacy_ingest.*"]
disallow_untyped_defs = false
disallow_incomplete_defs = false
disallow_untyped_calls = false
```

**The strictness ratchet:** add modules to the `strict` island over time; never remove them. This prevents typing debt from accumulating.

### Gradual typing adoption strategy

1. Start with `mypy --ignore-missing-imports` — no false positives from untyped deps
2. Add stubs or `py.typed` for dependencies gradually
3. Introduce strict islands module by module — each island is permanent
4. Use `# type: ignore[import]` as a quarantine, not a permanent solution
5. Track typing debt: count of `type: ignore` comments in CI

In [7]:
# Demonstrate what a stub file looks like in-notebook

stub_content = '''
# legacy_ingest/__init__.pyi
# Stub for an untyped legacy ingest library

def fetch_batch(source: str) -> list[dict[str, object]]: ...
def push_record(record: dict[str, object], destination: str) -> bool: ...
'''

pyproject_snippet = '''
[tool.mypy]
strict = true
enable_error_code = ["deprecated"]

[[tool.mypy.overrides]]
module = ["legacy_ingest", "legacy_ingest.*"]
disallow_untyped_defs = false
disallow_incomplete_defs = false
disallow_untyped_calls = false
'''

print("Stub file content:")
print(stub_content)
print("mypy config:")
print(pyproject_snippet)

Stub file content:

# legacy_ingest/__init__.pyi
# Stub for an untyped legacy ingest library

def fetch_batch(source: str) -> list[dict[str, object]]: ...
def push_record(record: dict[str, object], destination: str) -> bool: ...

mypy config:

[tool.mypy]
strict = true
enable_error_code = ["deprecated"]

[[tool.mypy.overrides]]
module = ["legacy_ingest", "legacy_ingest.*"]
disallow_untyped_defs = false
disallow_incomplete_defs = false
disallow_untyped_calls = false



---
## 6. Typing Tooling, CI, and Developer Experience

### mypy vs pyright

```
┌────────────────────────────────┬──────────────────────────────────┐
│             mypy               │            pyright               │
├────────────────────────────────┼──────────────────────────────────┤
│ The reference implementation   │ Microsoft's checker (used in     │
│                                │ Pylance / VS Code)               │
│ Python-based, extensible       │                                  │
│                                │ Faster, more incremental         │
│ Largest plugin ecosystem       │                                  │
│                                │ Often stricter on edge cases     │
│ Better at gradual migration    │                                  │
│                                │ Excellent IDE integration        │
│ pyproject.toml configuration   │                                  │
└────────────────────────────────┴──────────────────────────────────┘
```

**Emerging checkers:** `ty` (Astral, Rust-based) and `pyrefly` (Meta, Rust-based) — both faster than mypy/pyright but newer.

### Typing gates in CI

```yaml
# .github/workflows/ci.yml snippet
- name: Type-check
  run: mypy --strict src/
```

The CI break pattern: a type error blocks the build even when the code seems correct.

```python
# store.get_readings() returns Reading | None with latest=True
reading = store.get_readings(DeviceId(device_id), latest=True)
return reading.temperature  # ERROR: reading may be None
# Fix:
if reading is None:
    return None
return reading.temperature
```

### Testing the types themselves with `assert_type`

`assert_type(expr, T)` fails at type-check time if `expr` is not inferred as `T`. These are static tests — they verify what the checker *infers*, not what runs.

```python
# typetests/test_types.py
from typing import assert_type
from telemetry_hub.models import Reading, Summary, DeviceId
from telemetry_hub.store import ReadingStore
from telemetry_hub.pipeline import Pipeline

def check_get_readings_all(store: ReadingStore, device: DeviceId) -> None:
    assert_type(store.get_readings(device), list[Reading])

def check_get_readings_latest(store: ReadingStore, device: DeviceId) -> None:
    assert_type(store.get_readings(device, latest=True), Reading | None)

def check_pipeline_composition() -> None:
    from telemetry_hub.pipeline import validate, parse
    from collections.abc import Iterable
    assert_type(
        Pipeline(validate).then(parse),
        Pipeline[Iterable[object], list[Reading]]
    )
```

**`reveal_type`** is the interactive companion — the checker prints the inferred type without failing:

```python
reveal_type(store.get_readings(device))  # Revealed type: list[Reading]
```

In [8]:
# Demonstrate assert_type at runtime (it's a no-op there; the check is static)
from typing import assert_type

x: int = 42
assert_type(x, int)   # passes statically and at runtime
print("assert_type passed at runtime (it's always a no-op at runtime)")

# reveal_type is a mypy/pyright magic function — not available at runtime.
# Use it in .py files you run through the checker, not in notebooks.
# Example (run via: mypy script.py):
#
#   reveal_type(x)  # → Revealed type is "builtins.int"
#
# At runtime you can inspect the inferred annotation via __annotations__:
def annotated(val: int) -> None:
    pass

print("Annotation on parameter:", annotated.__annotations__)

assert_type passed at runtime (it's always a no-op at runtime)
Annotation on parameter: {'val': <class 'int'>, 'return': None}


---
## 7. Designing an Annotation-Driven Framework

### The annotation-driven pattern

> Annotations as a declarative DSL in ordinary Python syntax.

The pattern behind pydantic, FastAPI, and dataclasses:
1. You declare structure with annotations — fields, types, parameters
2. The framework reads them back at runtime
3. One declaration, many behaviors: validation, serialization, injection, code generation

### `Annotated[T, metadata]` — the extension point

```python
from typing import Annotated
temperature: Annotated[float, Range(-50.0, 150.0)]
```

- **Checker view:** `Annotated[float, Range(...)]` is just `float` — static checks are unchanged
- **Runtime view:** `get_args()` returns `(float, Range(-50.0, 150.0))` — metadata is accessible
- Both views coexist on one annotation — no new syntax
- This is the read-back mechanism behind pydantic `Field` and FastAPI `Depends`

### Approaches that predate `Annotated`

```
┌──────────────────────┬──────────────────────┬──────────────────────┐
│ Stringly-typed config│ Decorator arguments  │ External schema files│
├──────────────────────┼──────────────────────┼──────────────────────┤
│ Fields mirrored in   │ Every rule piled into│ Same drift, plus a   │
│ dicts or config files│ a decorator call     │ build step           │
│                      │                      │                      │
│ Drifts from the code │ Far from the fields  │ Annotated keeps the  │
│ it describes         │ it governs           │ rule on the field    │
│                      │                      │ that owns it         │
└──────────────────────┴──────────────────────┴──────────────────────┘
```

### `@dataclass_transform` — teaching the checker about generated code

Checkers know `@dataclass` as a built-in special case. A custom base class that generates `__init__` earns no such trust by default. `@dataclass_transform` marks a base class as dataclass-like so subclasses get checked constructors and editor completion.

### Annotation-driven dispatch

Parameter annotations are the routing table and the dependency list:
- Each new model no longer needs its own `isinstance` chain
- The same mechanism behind FastAPI's endpoints
- `register()` reads the function's annotations at decoration time
- `dispatch()` looks up the handler by payload type and injects dependencies

In [9]:
# Full annotation-driven framework
from __future__ import annotations
import dataclasses
from dataclasses import dataclass, field
from typing import Annotated, Any, Protocol, get_type_hints, get_origin, get_args, runtime_checkable
from typing import dataclass_transform


# ── Constraint protocol ──────────────────────────────────────────────

@runtime_checkable
class Constraint(Protocol):
    def check(self, value: object) -> str | None: ...


@dataclass(frozen=True)
class Range:
    min: float
    max: float

    def check(self, value: object) -> str | None:
        if not isinstance(value, (int, float)):
            return f"expected number, got {type(value).__name__}"
        if not (self.min <= value <= self.max):
            return f"{value} not in range [{self.min}, {self.max}]"
        return None


@dataclass(frozen=True)
class MaxLen:
    limit: int

    def check(self, value: object) -> str | None:
        if not isinstance(value, (str, bytes, list)):
            return None  # not applicable
        if len(value) > self.limit:
            return f"length {len(value)} exceeds max {self.limit}"
        return None


# ── Validation engine ────────────────────────────────────────────────

def _validate_field(path: str, value: object, annotation: object) -> list[str]:
    origin = get_origin(annotation)
    if origin is Annotated:
        base, *metadata = get_args(annotation)
        base_errors = _validate_field(path, value, base)
        if base_errors:
            return base_errors
        return [
            f"{path}: {message}"
            for item in metadata
            if isinstance(item, Constraint)
            and (message := item.check(value)) is not None
        ]
    return []


def validate(model_instance: object) -> list[str]:
    """Validate a model instance using Annotated constraints."""
    model_type = type(model_instance)
    errors: list[str] = []
    try:
        hints = get_type_hints(model_type, include_extras=True)
    except Exception:
        return []
    for field_name, annotation in hints.items():
        value = getattr(model_instance, field_name, None)
        errors.extend(_validate_field(field_name, value, annotation))
    return errors


# ── Model registry and @dataclass_transform ──────────────────────────

MODEL_REGISTRY: dict[str, type] = {}
HANDLER_REGISTRY: dict[type, Any] = {}
DEPENDENCIES: dict[Any, object] = {}


@dataclass_transform(frozen_default=True, field_specifiers=(field,))
class TelemetryModel:
    """Base class for all telemetry models. Subclasses become frozen dataclasses."""

    def __init_subclass__(cls, *, frozen: bool = True, **kwargs: Any) -> None:
        super().__init_subclass__(**kwargs)
        dataclass(frozen=frozen)(cls)
        MODEL_REGISTRY[cls.__name__] = cls


# ── Domain models ────────────────────────────────────────────────────

DeviceId = NewType("DeviceId", str)  # reuse from section 1

class SensorReading(TelemetryModel):
    device_id: Annotated[str, MaxLen(64)]
    temperature: Annotated[float, Range(-50.0, 150.0)]
    timestamp: str


class GpsReading(TelemetryModel):
    device_id: Annotated[str, MaxLen(64)]
    latitude: Annotated[float, Range(-90.0, 90.0)]
    longitude: Annotated[float, Range(-180.0, 180.0)]
    timestamp: str


# ── Annotation-driven dispatch ───────────────────────────────────────

from dataclasses import dataclass as dc

@dc
class Handler:
    fn: Any
    dependencies: dict[str, Any]


def register(fn: Any) -> Any:
    """Read fn's annotations: first param is the model type, rest are injected deps."""
    try:
        annotations = get_type_hints(fn)
    except Exception:
        annotations = fn.__annotations__.copy()
    annotations.pop("return", None)
    items = list(annotations.items())
    if not items:
        raise TypeError(f"{fn.__name__} needs at least one parameter (the model)")
    _, model_type = items[0]
    dependencies = {name: ann for name, ann in items[1:]}
    HANDLER_REGISTRY[model_type] = Handler(fn, dependencies)
    return fn


def provide(dependency_type: Any, instance: object) -> None:
    DEPENDENCIES[dependency_type] = instance


def dispatch(model: TelemetryModel) -> None:
    handler = HANDLER_REGISTRY.get(type(model))
    if handler is None:
        raise LookupError(f"No handler for {type(model).__name__}")
    kwargs = {
        name: DEPENDENCIES[ann]
        for name, ann in handler.dependencies.items()
    }
    handler.fn(model, **kwargs)


# ── Usage ────────────────────────────────────────────────────────────

class ConsoleSensorSink:
    def write(self, reading: SensorReading) -> None:
        print(f"  [sensor] device={reading.device_id} temp={reading.temperature}°C")

class ConsoleGpsSink:
    def write(self, reading: GpsReading) -> None:
        print(f"  [gps]    device={reading.device_id} lat={reading.latitude} lon={reading.longitude}")


@register
def handle_sensor(reading: SensorReading, sink: ConsoleSensorSink) -> None:
    # Validate before writing
    errors = validate(reading)
    if errors:
        print(f"  [validation errors] {errors}")
        return
    sink.write(reading)

@register
def handle_gps(reading: GpsReading, sink: ConsoleGpsSink) -> None:
    errors = validate(reading)
    if errors:
        print(f"  [validation errors] {errors}")
        return
    sink.write(reading)


# Wire dependencies
provide(ConsoleSensorSink, ConsoleSensorSink())
provide(ConsoleGpsSink, ConsoleGpsSink())

# Dispatch
print("Dispatching valid readings:")
dispatch(SensorReading(device_id="dev-01", temperature=22.5, timestamp="2024-01-01"))
dispatch(GpsReading(device_id="gps-01", latitude=48.2, longitude=16.3, timestamp="2024-01-01"))

print("\nDispatching invalid reading (temperature out of range):")
dispatch(SensorReading(device_id="dev-99", temperature=999.0, timestamp="2024-01-01"))

print("\nModel registry:", list(MODEL_REGISTRY.keys()))

Dispatching valid readings:
  [sensor] device=dev-01 temp=22.5°C
  [gps]    device=gps-01 lat=48.2 lon=16.3

Dispatching invalid reading (temperature out of range):
  [validation errors] ['temperature: 999.0 not in range [-50.0, 150.0]']

Model registry: ['SensorReading', 'GpsReading']


---
## Summary: Key Concepts and Quick Reference

### Type system architecture

| Layer | Role |
|-------|------|
| Annotations | Syntax only — parsed, attached to objects, never enforced |
| Type checkers (mypy/pyright) | All semantics and compatibility rules live here |
| Runtime | Stores annotations; never enforces them |

### Domain modeling tools

| Tool | Use when |
|------|----------|
| `NewType` | You need a distinct identity on an existing type (e.g. `DeviceId`) |
| `TypedDict` | Data that lives as a `dict` (wire payloads, JSON shapes) |
| `Literal` | A closed set of allowed values |
| `TypeIs[T]` | A user-written narrowing function that proves its arg is `T` |
| `@dataclass(frozen=True)` | Immutable domain objects after the parse boundary |

### Generic patterns

| Pattern | Syntax |
|---------|--------|
| Generic class (PEP 695) | `class Pipeline[TIn, TOut = TIn]:` |
| Generic function | `def first[T](items: list[T]) -> T:` |
| Bounded type var | `[T: Comparable]` — accepts bound and subtypes |
| Constrained type var | `[T: (str, bytes)]` — exactly these types |
| Signature-preserving decorator | `def decorator[**P, R](fn: Callable[P, R]) -> Callable[P, R]` |
| Overloaded API | `@overload` — each call shape gets its own return type |

### Protocol vs ABC

| | Protocol | ABC |
|-|----------|-----|
| Typing style | Structural (duck typing) | Nominal (inheritance) |
| Implementor needs | Nothing — just the right methods | Must inherit the ABC |
| Coupling | None | Coupled to import tree |
| `isinstance` | With `@runtime_checkable` (checks method presence only) | Always |

### Annotation-driven framework building blocks

| Tool | Purpose |
|------|---------|
| `Annotated[T, metadata]` | Attach constraint/metadata to a type without changing static behavior |
| `get_type_hints(cls, include_extras=True)` | Resolve annotations including `Annotated` extras |
| `get_origin(ann)` | Get `Annotated`, `list`, `Literal`, `Union`, etc. |
| `get_args(ann)` | Get the arguments — for `Annotated[float, Range(...)]`: `(float, Range(...))` |
| `@dataclass_transform` | Tell the checker a base class generates dataclass-like code |
| `annotationlib` (3.14+) | `Format.VALUE`, `Format.FORWARDREF`, `Format.STRING` |

### CI typing gates

```bash
# Run mypy in strict mode — fails the build on any type error
mypy --strict src/

# Run type tests (assert_type checks)
mypy typetests/
```

### Modern vs legacy annotation spellings

| Modern (3.10+) | Legacy |
|----------------|--------|
| `X \| None` | `Optional[X]` |
| `list[str]` | `List[str]` |
| `dict[str, int]` | `Dict[str, int]` |
| `type Status = Literal[...]` | `Status = TypeAlias = Literal[...]` |
| `TypeIs[T]` | `TypeGuard[T]` (broader) |
| `class Pipeline[T]` | `T = TypeVar("T"); class Pipeline(Generic[T])` |